In [18]:
import torch
import torch.nn as nn
import math
import random
from d2l import torch as d2l
from torch.nn import functional as F

In [19]:
import os
import requests
import hashlib
import zipfile
import collections
import re
import torch

def download_time_machine():
    url = 'http://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt'
    cache_dir = os.path.join('..', 'data')
    os.makedirs(cache_dir, exist_ok=True)
    fname = os.path.join(cache_dir, 'timemachine.txt')
    if not os.path.exists(fname):
        print("正在下载 timemachine.txt ...")
        r = requests.get(url)
        with open(fname, 'wb') as f:
            f.write(r.content)
        print("下载完成！")
    with open(fname, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # 转小写，只保留字母和空格
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

lines = download_time_machine()
print("前 3 行:", lines[:3])
print("总行数:", len(lines))


class Vocab:
    """简易词汇表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        # 统计词频
        counter = collections.Counter(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        # 索引到词元的列表
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):
        return 0  # <unk> 的索引


# 把所有行拼成一个长字符列表
corpus_chars = [char for line in lines for char in line]
vocab = Vocab(corpus_chars, min_freq=1, reserved_tokens=['<pad>'])
print("词汇表大小:", len(vocab))
print("前 10 个 token:", vocab.idx_to_token[:10])

# ---------- 3. 数值化整个语料库 ----------
corpus = [vocab[char] for char in corpus_chars]
print("语料库长度:", len(corpus))
print("前 20 个索引:", corpus[:20])

# ---------- 4. 顺序采样数据迭代器 ----------
def seq_data_iter_sequential(corpus, batch_size, num_steps):
    """顺序分区采样"""
    offset = random.randint(0, num_steps)
    num_tokens = ((len(corpus) - offset - 1) // batch_size) * batch_size
    Xs = torch.tensor(corpus[offset: offset + num_tokens])
    Ys = torch.tensor(corpus[offset + 1: offset + 1 + num_tokens])
    Xs = Xs.reshape(batch_size, -1)
    Ys = Ys.reshape(batch_size, -1)
    num_batches = Xs.shape[1] // num_steps
    for i in range(0, num_steps * num_batches, num_steps):
        X = Xs[:, i: i + num_steps]
        Y = Ys[:, i: i + num_steps]
        yield X, Y

def seq_data_iter_random(corpus, batch_size, num_steps):
    """随机采样"""
    corpus = corpus[random.randint(0, num_steps - 1):]
    num_subseqs = (len(corpus) - 1) // num_steps
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    random.shuffle(initial_indices)
    def data(pos):
        return corpus[pos: pos + num_steps]
    num_batches = num_subseqs // batch_size
    for i in range(0, batch_size * num_batches, batch_size):
        initial_indices_per_batch = initial_indices[i: i + batch_size]
        X = [data(j) for j in initial_indices_per_batch]
        Y = [data(j + 1) for j in initial_indices_per_batch]
        yield torch.tensor(X), torch.tensor(Y)

# 构造迭代器
batch_size, num_steps = 32, 35
train_iter = seq_data_iter_sequential(corpus, batch_size, num_steps)
rnn_layer = nn.RNN(len(vocab), num_hiddens)


前 3 行: ['the time machine by h g wells', '', '']
总行数: 3221
词汇表大小: 29
前 10 个 token: ['<unk>', '<pad>', ' ', 'e', 't', 'a', 'i', 'n', 'o', 's']
语料库长度: 170580
前 20 个索引: [4, 10, 3, 2, 4, 6, 14, 3, 2, 14, 5, 16, 10, 6, 7, 3, 2, 22, 20, 2]


In [20]:
num_hiddens=256

In [21]:
class RNNModel(nn.Module):
    def __init__(self, rnn_layer, vocab_size, **kwargs):
        super(RNNModel, self).__init__(**kwargs)
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size         
        if not self.rnn.bidirectional:                   
            self.num_directions = 1
            self.linear = nn.Linear(self.num_hiddens, self.vocab_size)
        else:
            self.num_directions = 2                      
            self.linear = nn.Linear(self.num_hiddens * 2, self.vocab_size)  # 修正4

    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size)
        X = X.to(torch.float32)
        Y, state = self.rnn(X, state)
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, device, batch_size=1):
        if not isinstance(self.rnn, nn.LSTM):
            return torch.zeros(
                (self.num_directions * self.rnn.num_layers,
                 batch_size,
                 self.num_hiddens),
                device=device
            )
        else:
            return (
                torch.zeros(
                    (self.num_directions * self.rnn.num_layers,
                     batch_size,
                     self.num_hiddens),
                    device=device
                ),
                torch.zeros(
                    (self.num_directions * self.rnn.num_layers,
                     batch_size,
                     self.num_hiddens),
                    device=device
                )
            )